# Introduction to Python and data analysis
Data Science for Business · Andreas Goethals · 2026–2027

We use section 1 for the first Colab demonstration, then sections 2–4 for the Sales example and Gemini.

Dataset: `Sales Demonstration Dataset.csv`. Assignment 1 has its own notebook, `A1_Notebook.ipynb`.


## 1. First steps in Google Colab
### Getting started and connecting
1. In [Google Drive](https://drive.google.com/), choose **New > File upload** and upload this `.ipynb` file.
2. Right-click it and choose **Open with > Google Colaboratory**.
3. Click **Connect**, wait until connected, then run a code cell with its play button or **Shift + Enter**.

The **runtime** is a Python session on Google's servers. It executes cells and keeps variables in memory. Use a standard CPU for today and Assignment 1.

Text cells use **Markdown**: `## Heading` makes a heading and `**bold**` makes bold text. Code cells run Python and show their output below. Double-click a text cell to edit it.


In [ ]:
print("Hello, world!")

### Values and variables
A variable stores a value for later use. Text needs quotes; numbers and `True` do not. Comments start with `#`. Names are case-sensitive.


In [ ]:
name = "Andreas"       # text
customers = 80         # whole number
sale_rate = 0.25       # decimal
is_ready = True        # true / false

print(name, customers, sale_rate, is_ready)

### Variables between cells
The next cell uses `name` from the previous one. After editing a value, rerun that cell and any dependent cells.

Restarting the runtime clears variables. Saved outputs may remain visible, so rerun cells from the top.


In [ ]:
print(name)  # This value comes from the previous cell.

### Calculations and output
`/` divides numbers and `print()` displays results. An f-string inserts values into text; `.0%` displays a fraction as a percentage.


In [ ]:
customers = 80
sales = 20
rate = sales / customers
print(rate)
print(f"Sale rate: {rate:.0%}")

### Lists and functions
A list stores values in order; positions start at zero.

A function groups reusable instructions. Here, `sum` adds the values, `len` counts them and `return` sends back the result. Indentation marks the function body.


In [ ]:
def mean(numbers):
    return sum(numbers) / len(numbers)

numbers = [20, 30, 40]
print("First value:", numbers[0])
print("Mean:", mean(numbers))

## 2. Load and inspect the data
In Colab, open **Files** (the folder icon), click upload and select `Sales Demonstration Dataset.csv`. Wait until it appears.

Python looks in its working folder, normally `/content` in Colab. If needed, use the file menu's **Copy path** and update `DATA_PATH`. Locally, keep the CSV in the working folder.

`import` loads libraries. `pd.read_csv` reads the CSV into a pandas table; `head()` displays its first rows. Opening the notebook from Drive does not load the CSV.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.model_selection import train_test_split
from IPython.display import display

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

In [ ]:
DATA_PATH = Path("Sales Demonstration Dataset.csv")

df = pd.read_csv(DATA_PATH)

display(df.head())

## 3. Define the outcome and split the data
Strip the spaces from `" Y"` and `" N"`, then encode sale as 1 and no sale as 0.

Set aside 30% for testing before exploring outcome relationships. `stratify` preserves class proportions; the random seed makes the split reproducible. Use training rows for the analysis below.

Summarise the training rows by region for the Gemini demo.


In [ ]:
sale_labels = df["sale"].str.strip()
assert set(sale_labels.unique()) == {"N", "Y"}, "Unexpected sale labels"
data = df.assign(sale_flag=sale_labels.map({"N": 0, "Y": 1}))

SEED = 2026
train, test = train_test_split(
    data, test_size=0.30, random_state=SEED, stratify=data["sale_flag"]
)
assert set(train.index).isdisjoint(test.index)
print(f"Training rows: {len(train):,}; test rows: {len(test):,}")

region_summary = (
    train.groupby("region")["sale_flag"]
    .agg(n="size", sales="sum", sale_rate="mean")
    .sort_values("sale_rate", ascending=False)
)
display(region_summary)


## 4. Gemini inside the notebook
Run sections 2–3, then open the **Gemini spark icon** in Colab. No API key is needed. Try this prompt:

> Use `region_summary`, indexed by region, with columns `n`, `sales` and `sale_rate`. Write a short Matplotlib cell plotting sale rates as percentages, with each bar labelled by `n`. Keep the existing data and split. Suggest code and explain it briefly.

Review the suggestion, paste it into the next cell and run it. If Gemini cannot see the table, include its displayed values in your prompt. Use the worked cell below if Gemini is unavailable.


In [ ]:
# Live demo: paste the reviewed Gemini plotting code here.
# During class, run this cell OR the worked plotting cell below.

### Worked example
Use this plotting cell as a fallback or comparison.


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.1))
bars = ax.bar(region_summary.index, region_summary["sale_rate"], color="#1B8DAD")
ax.bar_label(bars, labels=[f"n = {n:,}" for n in region_summary["n"]], padding=4)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set(xlabel="Region", ylabel="Sale rate", title="Sales by region in the training sample")
ax.set_ylim(0, max(0.45, region_summary["sale_rate"].max() + 0.12))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

### Check and explain
Compare bar heights and sample-size labels with `region_summary`. A rate of 0.25 should appear as 25%.

Ask Gemini:

> Explain how the code formats percentages and adds sample-size labels.

Which region has the highest rate? Which contributes the most sales? Consider group size.

Return to the slides for Assignment 1.


## Assignment 1
Open `A1_Notebook.ipynb` from Drive, connect, then upload `South_German_Credit.csv`. Run the loading cells and locate Questions 1–3.

Follow `A1_Assignment.pdf` and complete `A1_Template.docx`. Submit the report PDF and completed notebook as specified on page 1.
